# Pittsburgh, PA — Land Value Tax Model

**City of Pittsburgh real estate levy only, 4:1 split-rate, all current exemptions preserved.**

Pittsburgh ran a graded property tax (land taxed at a higher rate than improvements) from 1913 until 2001, peaking near a 5.77:1 land:building ratio. The city flat-rated in 2001 and currently bills 8.06 mills against all assessed value uniformly. This model rebuilds Pittsburgh under a 4:1 split-rate while preserving the city's $15,000 owner-occupied homestead exclusion and all exempt-class designations in the assessor file.

| | |
|---|---|
| Geographic scope | City of Pittsburgh (Allegheny County wards 1–32, MUNICODE 101–132) |
| Levy modeled | City of Pittsburgh real estate tax only |
| Reform | Revenue-neutral 4:1 split-rate (land taxed 4× improvements) |
| Validation target | Gross levy on $20.35B taxable AV → ≈$164M (FY25 actual collections ≈$148M reflect ~90% collection rate) |
| Data sources | WPRDC Allegheny County master assessment file (attributes); Allegheny County GIS `Web_Parcels` MapServer (geometry) |

In [ ]:
import sys
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)

CITY_NAME = 'pittsburgh'
STATE_FIPS = '42'      # Pennsylvania
COUNTY_FIPS = '003'    # Allegheny County
MODEL_TYPE = 'split_rate_4to1'
LAND_IMPROVEMENT_RATIO = 4.0

CITY_MILLAGE = 8.06          # FY2025 City of Pittsburgh real estate millage
HOMESTEAD_EXCLUSION = 15000  # City of Pittsburgh owner-occupied homestead exclusion (off LOCALTOTAL)
OFFICIAL_TAXABLE_AV = 20_350_000_000  # FY2025 reported by Allegheny Institute / county
OFFICIAL_GROSS_LEVY = OFFICIAL_TAXABLE_AV * CITY_MILLAGE / 1000.0

DATA_DIR = Path('data')
ATTR_PATH = DATA_DIR / 'allegheny_county_master_file.csv'
GEOM_PATH = DATA_DIR / 'pittsburgh_geometry.parquet'

print(f"City millage: {CITY_MILLAGE} mills")
print(f"Homestead exclusion: ${HOMESTEAD_EXCLUSION:,}")
print(f"Official taxable AV: ${OFFICIAL_TAXABLE_AV:,}")
print(f"Implied gross levy: ${OFFICIAL_GROSS_LEVY:,.0f}")

## Section 2 — Load Parcel Data

**Attributes**: WPRDC Allegheny County master file (CSV, ~585K county-wide parcels). Filter to Pittsburgh wards (MUNICODE 101–132).

**Geometry**: Allegheny County `Web_Parcels` MapServer, served in EPSG:2272 (PA State Plane South) but requested in EPSG:4326 via `outSR`. Filtered to Pittsburgh PARIDs during download.

Join keys: WPRDC `PARID` ↔ GIS `PIN` (both 16-character parcel identifiers).

| Concept | Column |
|---|---|
| Land assessed value (city/school taxable) | `LOCALLAND` |
| Improvement assessed value (city/school taxable) | `LOCALBUILDING` |
| Total assessed value | `LOCALTOTAL` = `LOCALLAND` + `LOCALBUILDING` |
| Fair-market values (2012 base year, no reductions) | `FAIRMARKETLAND`, `FAIRMARKETBUILDING`, `FAIRMARKETTOTAL` |
| Tax status | `TAXCODE` ∈ {`T`, `E`, `P`} — taxable, exempt, PURTA |
| City homestead approved | `HOMESTEADFLAG` = `'HOM'` |
| Clean & Green preferential | `CLEANGREEN` = `'Y'` (already baked into LOCAL values) |
| Use code (200+ values) | `USECODE`, `USEDESC` |
| Broad class | `CLASS` ∈ {R, C, I, U, G, O, F}, `CLASSDESC` |
| Municipality (Pittsburgh wards) | `MUNICODE` 101–132 |

PA assessment ratio note: Allegheny County's predetermined ratio is 100% of base-year (2012) fair market value, so `LOCALTOTAL` is the taxable base directly — no fractional assessment ratio is applied. The county-wide reassessment lag is the reason for the diverging Common Level Ratio (~52.7% in 2025), but that affects appeals only, not the per-parcel tax bill.

In [ ]:
# Load attributes (WPRDC master file)
attr_cols = [
    'PARID', 'PROPERTYADDRESS', 'PROPERTYCITY', 'PROPERTYZIP',
    'MUNICODE', 'MUNIDESC', 'SCHOOLDESC', 'NEIGHCODE', 'NEIGHDESC',
    'TAXCODE', 'TAXDESC', 'OWNERDESC',
    'CLASS', 'CLASSDESC', 'USECODE', 'USEDESC', 'LOTAREA',
    'HOMESTEADFLAG', 'FARMSTEADFLAG', 'CLEANGREEN', 'ABATEMENTFLAG',
    'COUNTYBUILDING', 'COUNTYLAND', 'COUNTYTOTAL',
    'LOCALBUILDING', 'LOCALLAND', 'LOCALTOTAL',
    'FAIRMARKETBUILDING', 'FAIRMARKETLAND', 'FAIRMARKETTOTAL',
    'YEARBLT', 'STYLEDESC', 'GRADEDESC',
]
attr_dtypes = {
    'PARID': str, 'MUNICODE': str, 'USECODE': str, 'CLASS': str,
    'NEIGHCODE': str, 'TAXCODE': str, 'HOMESTEADFLAG': str,
    'CLEANGREEN': str, 'ABATEMENTFLAG': str, 'FARMSTEADFLAG': str,
}
attr = pd.read_csv(ATTR_PATH, usecols=attr_cols, dtype=attr_dtypes, low_memory=False)
attr['MUNICODE_INT'] = pd.to_numeric(attr['MUNICODE'], errors='coerce')
attr = attr[attr['MUNICODE_INT'].between(101, 132)].copy()
for col in ('COUNTYBUILDING', 'COUNTYLAND', 'COUNTYTOTAL',
            'LOCALBUILDING', 'LOCALLAND', 'LOCALTOTAL',
            'FAIRMARKETBUILDING', 'FAIRMARKETLAND', 'FAIRMARKETTOTAL',
            'LOTAREA'):
    attr[col] = pd.to_numeric(attr[col], errors='coerce').fillna(0).clip(lower=0)
print(f"Pittsburgh parcels (attributes): {len(attr):,}")
print(f"Of which TAXCODE='T' (taxable): {(attr['TAXCODE']=='T').sum():,}")
print(f"Homestead approved: {(attr['HOMESTEADFLAG']=='HOM').sum():,}")

# Load geometry
geom = gpd.read_parquet(GEOM_PATH)
geom = geom.rename(columns={'PIN': 'PARID'})
print(f"\nPittsburgh parcels (geometry): {len(geom):,}")

# Merge: attributes drive the join (some parcels lack polygons; that's OK for tax modeling)
gdf = geom.merge(attr, on='PARID', how='right')
gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs='EPSG:4326')
geom_match_pct = gdf.geometry.notna().mean() * 100
print(f"\nMerged parcels: {len(gdf):,}  ({geom_match_pct:.1f}% have polygons)")

## Section 3 — Classify and Validate

Map Allegheny's ~140 USECODE values to LVTShift's standard property categories, identify exempt parcels, and confirm taxable assessed value against the official Allegheny Institute figure ($20.35B for FY2025).

In [ ]:
CONDO_USECODES = {'050', '055', '056', '057', '118', '450', '550', '557', '558', '559'}

def categorize_pittsburgh(usecode):
    """Map Allegheny County USECODE → LVTShift PROPERTY_CATEGORY."""
    if pd.isna(usecode):
        return 'Other'
    c = str(usecode).strip().zfill(3)
    if c == '010': return 'Single Family Residential'
    if c in ('020', '030', '040'): return 'Small Multi-Family (2-4 units)'
    # 050 (residential condo), 055 (common area), 056 (condo developmental land),
    # 057 (condo garage units), 118 (condo common property), 450 (condo office
    # building units), 550 (commercial condo unit) all live in one Condominium
    # bucket since they share the same land-allocation pathology.
    if c in ('050', '055', '056', '057', '118', '450', '550'): return 'Condominium'
    if c in ('060', '070'): return 'Townhome / Rowhouse'
    if c in ('080', '090', '098', '099', '130', '131', '470', '471', '472', '473'): return 'Other Residential'
    if c in ('100', '111', '110', '300', '400'): return 'Vacant Land'
    if c in ('401', '402', '403', '557', '558', '559'): return 'Large Multi-Family (5+ units)'
    if c in ('404', '405', '406', '431', '432', '433'): return 'Mixed Use'
    if c in ('409', '410', '411', '412', '413', '415'): return 'Hotel'
    if c in ('420', '421', '422', '423', '424', '425', '426', '427', '429',
             '430', '434', '435', '437', '439', '440', '441', '444',
             '491', '492', '493', '494', '750'): return 'Retail / General Commercial'
    if c in ('442', '447', '448', '449'): return 'Office / Commercial Condo'
    if c in ('455', '456', '777'): return 'Transportation - Parking'
    if c in ('310', '320', '330', '340', '341', '350', '351', '352', '353',
             '370', '389', '399', '480', '482', '840', '850', '860'): return 'Industrial'
    if c in ('109', '220', '230'): return 'Agricultural'
    if c in ('600', '601', '602', '603', '605', '607', '609',
             '610', '620', '630', '640', '645', '650', '660', '670',
             '680', '685', '690', '700', '710', '720', '730',
             '418', '419', '451'): return 'Exempt'
    if c in ('452', '453', '454', '460', '461', '464', '465',
             '474', '488', '489', '490', '496', '499', '481'): return 'Other Commercial'
    return 'Other'

gdf['PROPERTY_CATEGORY'] = gdf['USECODE'].map(categorize_pittsburgh)

# Hard override: $0 improvement → Vacant Land regardless of use code.
# IMPORTANT: skip Condominium parcels in this override — they legitimately
# have $0 improvement at the unit level (e.g., parking-stall condos, common-area
# parcels, condo developmental land) but still belong to the Condominium category
# for the land-imputation pass that follows in the next cell.
vacant_override_categories = [
    'Single Family Residential', 'Small Multi-Family (2-4 units)',
    'Townhome / Rowhouse', 'Other Residential', 'Large Multi-Family (5+ units)',
    'Mixed Use', 'Retail / General Commercial', 'Office / Commercial Condo',
    'Transportation - Parking', 'Industrial', 'Other Commercial', 'Other']
vacant_override = (gdf['LOCALBUILDING'] <= 0) & gdf['PROPERTY_CATEGORY'].isin(vacant_override_categories)
gdf.loc[vacant_override, 'PROPERTY_CATEGORY'] = 'Vacant Land'

# Exempt flag for the model: TAXCODE=='E' (Exempt) OR PROPERTY_CATEGORY=='Exempt'
# PURTA ('P') parcels pay tax into a state fund — not city revenue — so we treat them as exempt for revenue-neutrality.
gdf['full_exmp'] = (gdf['TAXCODE'].isin(['E', 'P']) | (gdf['PROPERTY_CATEGORY'] == 'Exempt')).astype(int)

print('Category distribution:')
print(gdf['PROPERTY_CATEGORY'].value_counts())
print(f"\nFully exempt (TAXCODE=E/P or PROPERTY_CATEGORY=Exempt): {gdf['full_exmp'].sum():,}")
print(f"Taxable for split-rate model: {(gdf['full_exmp']==0).sum():,}")

# Diagnostic: what falls into 'Other' or 'Other Commercial'?
other_codes = gdf.loc[gdf['PROPERTY_CATEGORY'].isin(['Other', 'Other Commercial', 'Other Residential']),
                       ['USECODE', 'USEDESC']].value_counts().head(15)
if len(other_codes):
    print(f"\nTop USECODE values inside the residual 'Other*' buckets:")
    print(other_codes)


### Condo collapse to landlots, with multi-tier IR imputation

Allegheny County books **4,679 of 4,700 residential condos (USECODE 050) at $0 land** and the supposed common-property parcels (USECODE 118) at $0 too — the underlying land is missing from the assessment file. The same pathology hits commercial condo units (USECODE 550, 234 parcels), condo garage units (USECODE 057, 74 parcels), and the "condominium office building" code (USECODE 450, 78 parcels): roughly 300 of these 386 parcels also carry $0 land.

We do two things in this cell:

**1. Collapse condo units to landlots.** A condo building is one physical landlot — Pittsburgh just bills the 50 units in it separately. For LVT analysis we treat one building as one row. We group every condo-style parcel by `parid_root10` (the first 10 characters of PARID = block + section + lot, equivalent to Allegheny's MAPBLOCKLOT). 5,367 condo unit rows collapse to **442 landlot rows** (~12 units per building). Building values are summed; the count of homesteaded units is preserved so we still apply the right total homestead exclusion at the landlot level. The 137 non-condo parcels that happen to share `parid_root10` with a condo building (mostly townhomes) are left as separate rows.

**2. Three-tier IR fallback for land imputation.** Condo buildings live in 61P*-prefixed "condo-only" valuation neighborhoods that have *no* non-condo parcels for IR computation. A single city-wide fallback IR caused every fallback condo to land at the same Δ% (an artifact that pinned income-quintile medians at +2.9%). Instead we use:

1. **NEIGHCODE median IR** when the valuation neighborhood has ≥10 non-condo parcels
2. **MUNICODE (ward) median IR** when the neighborhood has no non-condo support — 32 wards, IRs from 0.588 to 0.898 (σ ≈ 0.074)
3. **City-wide median IR** (0.730) only as a last resort

The ward fallback gives condo buildings in different wards different imputed land shares, breaking the artificial concentration of identical Δ% values that distorted the income-quintile chart.


In [ ]:
# Collapse condos to landlots, then impute land at the building level using a
# 3-tier IR fallback (NEIGHCODE → MUNICODE → city). Replaces per-unit
# imputation which produced identical Δ% across thousands of condo units
# and pinned income-quintile medians at +2.9%.

CONDO_USECODES_IMPUTE = {'050', '055', '056', '057', '118', '450', '550', '557', '558', '559'}
LAND_FLOOR = 100              # treat LOCALLAND per unit <= $100 as "no land assessment"
IR_FLOOR, IR_CEIL = 0.30, 0.95

# 1) Multi-level non-condo IR.
non_condo_mask = (
    (~gdf['USECODE'].isin(CONDO_USECODES_IMPUTE))
    & (gdf['TAXCODE'] == 'T')
    & (gdf['LOCALLAND'] > 0)
    & (gdf['LOCALBUILDING'] > 0)
)
nc = gdf.loc[non_condo_mask, ['NEIGHCODE', 'MUNICODE', 'LOCALLAND', 'LOCALBUILDING']].copy()
nc['IR'] = nc['LOCALBUILDING'] / (nc['LOCALLAND'] + nc['LOCALBUILDING'])

nbhd_ir = nc.groupby('NEIGHCODE').agg(nbhd_ir=('IR', 'median'), nbhd_n=('IR', 'count')).reset_index()
muni_ir = nc.groupby('MUNICODE').agg(muni_ir=('IR', 'median'), muni_n=('IR', 'count')).reset_index()
city_ir = float(nc['IR'].median())

# Drop NEIGHCODE IRs with thin support so they fall through to MUNICODE.
nbhd_ir.loc[nbhd_ir['nbhd_n'] < 10, 'nbhd_ir'] = np.nan

print(f"IR fallback hierarchy:")
print(f"  city-wide median IR: {city_ir:.3f}  (⇒ land share {1-city_ir:.1%})")
print(f"  NEIGHCODE IRs with ≥10 non-condo parcels: {nbhd_ir['nbhd_ir'].notna().sum()} / {len(nbhd_ir)}")
print(f"  MUNICODE (ward) IRs: {len(muni_ir)} wards, range "
      f"{muni_ir['muni_ir'].min():.3f}–{muni_ir['muni_ir'].max():.3f}, σ={muni_ir['muni_ir'].std():.3f}")

# 2) Split condo vs non-condo rows. Group condos by parid_root10 (= MAPBLOCKLOT).
gdf['parid_root10'] = gdf['PARID'].str[:10]
condo_mask = gdf['USECODE'].isin(CONDO_USECODES_IMPUTE)
condo_rows = gdf[condo_mask].copy()
non_condo_rows = gdf[~condo_mask].copy()
print(f"\nCondo rows pre-collapse: {len(condo_rows):,}")
print(f"Non-condo rows: {len(non_condo_rows):,}")
print(f"Unique condo landlots (parid_root10): {condo_rows['parid_root10'].nunique()}")

# 3) Aggregate condo rows to landlots.
SUM_COLS = ['LOCALBUILDING', 'LOCALLAND',
            'COUNTYBUILDING', 'COUNTYLAND', 'COUNTYTOTAL',
            'FAIRMARKETBUILDING', 'FAIRMARKETLAND', 'FAIRMARKETTOTAL',
            'LOTAREA']
FIRST_COLS = ['MUNICODE', 'MUNIDESC', 'MUNICODE_INT', 'NEIGHCODE', 'NEIGHDESC',
              'SCHOOLDESC', 'TAXCODE', 'TAXDESC', 'USECODE', 'USEDESC',
              'CLASS', 'CLASSDESC', 'PROPERTYADDRESS', 'PROPERTYCITY', 'PROPERTYZIP',
              'OWNERDESC', 'CLEANGREEN', 'ABATEMENTFLAG', 'FARMSTEADFLAG',
              'YEARBLT', 'STYLEDESC', 'GRADEDESC', 'PARID', 'PROPERTY_CATEGORY']
agg_spec = {c: 'sum' for c in SUM_COLS if c in condo_rows.columns}
agg_spec.update({c: 'first' for c in FIRST_COLS if c in condo_rows.columns})

condo_collapsed = condo_rows.groupby('parid_root10', as_index=False).agg(agg_spec)
condo_collapsed['n_units'] = condo_rows.groupby('parid_root10').size().reindex(condo_collapsed['parid_root10']).values
condo_collapsed['homestead_units'] = (
    condo_rows.assign(_h=(condo_rows['HOMESTEADFLAG'] == 'HOM').astype(int))
    .groupby('parid_root10')['_h'].sum()
    .reindex(condo_collapsed['parid_root10']).values
)
condo_collapsed['HOMESTEADFLAG'] = np.where(condo_collapsed['homestead_units'] > 0, 'HOM', None)
condo_collapsed['LOCALTOTAL'] = condo_collapsed['LOCALLAND'] + condo_collapsed['LOCALBUILDING']

# Geometry: union polygons per landlot via dissolve.
condo_gdf_geom = condo_rows[['parid_root10', 'geometry']].dropna(subset=['geometry'])
if len(condo_gdf_geom):
    dissolved = condo_gdf_geom.dissolve(by='parid_root10').reset_index()
else:
    dissolved = pd.DataFrame({'parid_root10': [], 'geometry': []})
condo_collapsed = condo_collapsed.merge(dissolved, on='parid_root10', how='left')

# 4) Apply 3-tier IR fallback to collapsed landlots.
condo_collapsed = condo_collapsed.merge(nbhd_ir[['NEIGHCODE', 'nbhd_ir']], on='NEIGHCODE', how='left')
condo_collapsed = condo_collapsed.merge(muni_ir[['MUNICODE', 'muni_ir']], on='MUNICODE', how='left')
condo_collapsed['effective_ir'] = (
    condo_collapsed['nbhd_ir']
    .fillna(condo_collapsed['muni_ir'])
    .fillna(city_ir)
    .clip(IR_FLOOR, IR_CEIL)
)

# Mark which tier each row used (for QA + the validation summary).
condo_collapsed['ir_source'] = np.where(
    condo_collapsed['nbhd_ir'].notna(), 'NEIGHCODE',
    np.where(condo_collapsed['muni_ir'].notna(), 'MUNICODE', 'CITY'),
)
print(f"\nIR source for the {len(condo_collapsed)} collapsed landlots:")
print(condo_collapsed['ir_source'].value_counts().to_string())

# Impute building-level land where the collapsed landlot has near-zero land.
needs_imp = (
    (condo_collapsed['LOCALLAND'] <= condo_collapsed['n_units'] * LAND_FLOOR)
    & (condo_collapsed['LOCALBUILDING'] > 0)
)
imputed_land = condo_collapsed['LOCALBUILDING'] * (1.0 - condo_collapsed['effective_ir']) / condo_collapsed['effective_ir']
condo_collapsed.loc[needs_imp, 'LOCALLAND'] = imputed_land[needs_imp].round(0)
condo_collapsed['LOCALTOTAL'] = condo_collapsed['LOCALLAND'] + condo_collapsed['LOCALBUILDING']
condo_collapsed['land_imputed'] = needs_imp.astype(int)

added_land = condo_collapsed.loc[needs_imp, 'LOCALLAND'].sum()
print(f"\nLand value reallocated to collapsed condo landlots: ${added_land/1e6:,.1f}M")

# 5) Mark non-condo rows with unit counts so the homestead exclusion math
#    is uniform downstream.
non_condo_rows = non_condo_rows.copy()
non_condo_rows['n_units'] = 1
non_condo_rows['homestead_units'] = (non_condo_rows['HOMESTEADFLAG'] == 'HOM').astype(int)
non_condo_rows['land_imputed'] = 0
non_condo_rows['effective_ir'] = np.nan
non_condo_rows['ir_source'] = 'NA'

# Drop the helper cols on collapsed if any are missing in non_condo (avoid concat NaN cols).
common_cols = [c for c in condo_collapsed.columns if c in non_condo_rows.columns or c in ('n_units','homestead_units','land_imputed','effective_ir','ir_source')]
extra_cols = [c for c in condo_collapsed.columns if c not in non_condo_rows.columns]
for c in extra_cols:
    if c not in non_condo_rows.columns:
        non_condo_rows[c] = np.nan if c not in ('n_units','homestead_units','land_imputed') else 0

# 6) Recombine: non-condo rows + collapsed condo landlots.
gdf = pd.concat([non_condo_rows, condo_collapsed], ignore_index=True)
gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs='EPSG:4326')

print(f"\nPost-collapse parcel total: {len(gdf):,}  "
      f"(was {len(non_condo_rows) + len(condo_rows):,} pre-collapse)")
print(f"Citywide taxable AV (post-impute): "
      f"${gdf.loc[gdf['TAXCODE']=='T','LOCALTOTAL'].sum()/1e9:.3f}B")
print(f"Land share: "
      f"{gdf.loc[gdf['TAXCODE']=='T','LOCALLAND'].sum() / gdf.loc[gdf['TAXCODE']=='T','LOCALTOTAL'].sum() * 100:.1f}%")


## Section 4 — Current Tax Model

Pittsburgh applies its 8.06 mill operating millage to `LOCALTOTAL` (assessed value, no fractional ratio applied) net of:

- **$15,000 city homestead exclusion** for owner-occupied principal dwellings (`HOMESTEADFLAG == 'HOM'`)
- Exempt parcels (`TAXCODE == 'E'`) pay $0
- PURTA utility parcels (`TAXCODE == 'P'`) pay into a state fund — excluded from city revenue base

Not modeled (insufficient parcel-level signal):

- **Act 77 senior tax relief** — 30–40% city-tax discount for income-restricted owner-occupants aged 60+. The parcel file has no income flag, so this discount is not subtracted. Effect: model slightly overstates current tax for ~5–10% of homestead parcels.
- **LERTA / Act 42 / Act 77 building abatements** — only 175 Pittsburgh parcels have `ABATEMENTFLAG = 'Y'`, and that flag indicates a *county* abatement; corresponding city abatements are not separately flagged. Effect: small revenue overstatement on a tiny share of parcels.

Validation: the modeled taxable assessed value should match the FY2025 reported figure of $20.35B to within ~0.1%. The modeled gross levy ($164M) sits above FY2025 net collections ($148M) because the ~90% collection rate reflects delinquency, refunds, and successful appeals — none of which the parcel file exposes.

In [ ]:
# Compute taxable land / improvement / total.
#
# Homestead exclusion: $15K per HOMESTEAD-flagged owner-occupied unit. For
# collapsed condo landlots that holds n homesteaded units, the building gets
# n × $15K off its LOCALTOTAL. The `homestead_units` column was set by the
# collapse cell above; non-condo rows have homestead_units ∈ {0, 1}.
gdf['homestead_exclusion_amt'] = (HOMESTEAD_EXCLUSION
                                  * gdf['homestead_units'].fillna(0).astype(int))
gdf.loc[gdf['full_exmp'] == 1, 'homestead_exclusion_amt'] = 0

# Apportion the homestead deduction between land and building in proportion
# to their LOCAL assessed values, so the split-rate model receives consistent
# land vs. improvement components.
_total = gdf['LOCALTOTAL'].clip(lower=1)
_land_share = gdf['LOCALLAND'] / _total
_imp_share = gdf['LOCALBUILDING'] / _total

gdf['taxable_land_value'] = (gdf['LOCALLAND'] - _land_share * gdf['homestead_exclusion_amt']).clip(lower=0)
gdf['taxable_improvement_value'] = (gdf['LOCALBUILDING'] - _imp_share * gdf['homestead_exclusion_amt']).clip(lower=0)
gdf['taxable_total'] = gdf['taxable_land_value'] + gdf['taxable_improvement_value']

gdf['millage_rate'] = CITY_MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_total',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

modeled_av = gdf.loc[gdf['full_exmp'] == 0, 'taxable_total'].sum()
print(f"Modeled taxable AV:     ${modeled_av/1e9:.3f}B")
print(f"Official taxable AV:    ${OFFICIAL_TAXABLE_AV/1e9:.3f}B")
print(f"AV match gap:           {(modeled_av/OFFICIAL_TAXABLE_AV - 1)*100:+.2f}%")
print()
print(f"Modeled gross levy:     ${current_revenue:,.0f}")
print(f"Implied @ official AV:  ${OFFICIAL_GROSS_LEVY:,.0f}")
print(f"FY2025 collected (~90% rate): $148,000,000")
print(f"FY2025 budgeted:                $143,900,000")
print()
gap = (current_revenue / OFFICIAL_GROSS_LEVY - 1) * 100
print(f"Gap vs gross levy: {gap:+.2f}%")
assert abs(gap) < 5.0, f"Revenue gap {gap:.1f}% exceeds 5% threshold"


## Section 5 — 4:1 Split-Rate Model

Apply the LVTShift revenue-neutral solver. Excluded from the solve: parcels with `full_exmp = 1` (Allegheny exempt + PURTA + government/charitable use codes). All other parcels participate; their `new_tax` is set so total revenue equals the current city levy.

In [ ]:
taxable = gdf[gdf['full_exmp'] == 0].copy()
exempt = gdf[gdf['full_exmp'] == 1].copy()

target_revenue = taxable['current_tax'].sum()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=target_revenue,
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

# Recombine: exempt parcels carry zero tax (current and new)
for col in ('new_tax', 'tax_change', 'tax_change_pct'):
    if col not in exempt.columns:
        exempt[col] = 0.0
exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0

gdf = pd.concat([taxable, exempt]).sort_index()
gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs='EPSG:4326')

print(f"4:1 split-rate revenue-neutral millages:")
print(f"  Land millage:        {land_millage:.4f}  ({land_millage/CITY_MILLAGE:.2f}× current uniform rate)")
print(f"  Improvement millage: {improvement_millage:.4f}  ({improvement_millage/CITY_MILLAGE:.2f}× current uniform rate)")
print(f"  Ratio:               {land_millage/improvement_millage:.2f}:1")
print(f"  New revenue:         ${new_revenue:,.0f}")
print(f"  Target revenue:      ${target_revenue:,.0f}")
print(f"  Neutrality gap:      {(new_revenue/target_revenue-1)*100:+.4f}%")

In [ ]:
# Category summary table
category_summary = calculate_category_tax_summary(
    df=gdf[gdf['full_exmp'] == 0],
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Pittsburgh — 4:1 Split-Rate Tax Impact by Property Category')

## Section 6 — Exploration Chart

Quick visual of median tax change by category for in-notebook review. Headless render only — the standard 7-PNG report is generated in Section 7.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
med = gdf[gdf['full_exmp'] == 0].groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values()
colors = ['#c0504d' if v > 0 else '#4f81bd' for v in med.values]
med.plot.barh(ax=ax, color=colors)
ax.set_title(f'Pittsburgh — Median Tax Change % by Category (4:1 Split-Rate)')
ax.set_xlabel('Median % change')
ax.axvline(0, color='black', linewidth=0.7)
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print(f"Preview saved to {DATA_DIR / 'category_preview.png'}")

In [ ]:
# Census join — Allegheny County (42003) requires direct Layer 1 fetch.
# Background: lvt.census_utils.get_census_blockgroups_shapefile uses TIGERweb Layer 2
# ('2020 Census Blocks', 24,787 records for Allegheny). That request is server-rejected
# and the per-tract chunked fallback then has to walk 1,062 tracts — exceeding the cell
# timeout. Layer 1 is 'Census Block Groups' (1,062 records, maxRecord 100K) — fetches
# in a single query. We pair it with the in-package ACS demographics call which works fine.

import requests
import geopandas as gpd
from lvt.census_utils import get_census_data, match_to_census_blockgroups

CENSUS_CACHE = DATA_DIR / f'census_{STATE_FIPS}{COUNTY_FIPS}.parquet'

def _fetch_blockgroups(state_fips, county_fips):
    url = 'https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/Tracts_Blocks/MapServer/1/query'
    params = {
        'where': f"STATE='{state_fips}' AND COUNTY='{county_fips}'",
        'outFields': 'STATE,COUNTY,TRACT,BLKGRP,GEOID',
        'returnGeometry': 'true',
        'outSR': '4326',
        'f': 'geojson',
    }
    resp = requests.get(url, params=params, timeout=120)
    resp.raise_for_status()
    bgs = gpd.GeoDataFrame.from_features(resp.json()['features'], crs='EPSG:4326')
    bgs['std_geoid'] = bgs['STATE'] + bgs['COUNTY'] + bgs['TRACT'] + bgs['BLKGRP']
    return bgs

try:
    if CENSUS_CACHE.exists():
        census_bg = gpd.read_parquet(CENSUS_CACHE)
        print(f"Loaded {len(census_bg):,} block groups from cache")
    else:
        bg_geom = _fetch_blockgroups(STATE_FIPS, COUNTY_FIPS)
        print(f"Fetched {len(bg_geom):,} block group polygons from TIGERweb Layer 1")
        acs = get_census_data(STATE_FIPS + COUNTY_FIPS, year=2022)
        print(f"Fetched {len(acs):,} ACS demographic rows")
        census_bg = bg_geom.merge(
            acs[['std_geoid', 'median_income', 'total_pop', 'white_pop', 'black_pop',
                 'hispanic_pop', 'minority_pct', 'black_pct']],
            on='std_geoid', how='left',
        )
        census_bg = gpd.GeoDataFrame(census_bg, geometry='geometry', crs='EPSG:4326')
        census_bg.to_parquet(CENSUS_CACHE)
        print(f"Cached -> {CENSUS_CACHE}")

    gdf = match_to_census_blockgroups(gdf, census_bg)
    pct = gdf['std_geoid'].notna().mean() * 100
    print(f"Census join: {pct:.1f}% matched")
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        if _col not in gdf.columns:
            gdf[_col] = float('nan')


In [ ]:
# Export — gdf must have census columns by this point.
# Pass exempt_flag_col='full_exmp' so the standard export carries an accurate
# is_fully_exempt for downstream cross-city work.
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    exempt_flag_col='full_exmp',
)

# Standard report — 7 PNGs in analysis/reports/<city>/.
# Pass the taxable-only slice to the report so the Exempt category does NOT
# appear as a flat 0% bar in category_impact.png and ten_pct_share.png. The
# full out_df (with exempt rows) is still written to analysis/data/<city>.csv
# for cross-city analysis.
from lvt.viz import create_city_report
report_df = out_df[~out_df['is_fully_exempt']].copy()
print(f"Filtering for charts: {len(out_df):,} total → {len(report_df):,} taxable "
      f"(excluding {out_df['is_fully_exempt'].sum():,} exempt rows).")
create_city_report(report_df, CITY_NAME, show=False)
print('Done.')


## Validation Summary

| Check | Result |
|---|---|
| Parcel count | 137,467 rows after collapse (5,367 condo units → 442 landlots; 137,025 non-condo rows untouched) |
| Geometry coverage | 99.9% (landlot polygons dissolved from constituent condo units) |
| Condo land imputed | 442 landlots — $764.3M reallocated to buildings from absent common-property assessments |
| IR fallback distribution | 224 landlots used NEIGHCODE IR, 218 used MUNICODE (ward) IR, 0 fell back to city-wide median — pinning artifact eliminated |
| Taxable assessed value | $20.869B modeled vs. $20.350B official (FY2025) → **+2.55% gap** (model is intentionally above official because imputation restores land value the assessor omitted) |
| Gross levy | $168.2M modeled vs. $164.0M implied at official AV → revenue-neutral solver converges at 0.0000% |
| Census coverage | 99.9% of parcels matched to block groups (Allegheny: 1,062 BGs via TIGERweb Layer 1) |
| PNGs generated | 7 of 7 — Exempt category filtered out of category and ten-pct charts |
| Land millage | **18.28 mills** (2.27× current uniform rate) |
| Improvement millage | **4.57 mills** (0.57× current uniform rate) |
| City-wide taxable land share | **25.3%** after collapse + imputation |

### Per-category outcomes at 4:1 split-rate

| Category | n | Median Δ$ | Median Δ% |
|---|---|---|---|
| Vacant Land | 17,865 | $35 | **+126.8%** |
| Other Commercial | 984 | $124 | +79.7% |
| Transportation — Parking | 765 | $187 | +61.3% |
| Other Residential | 1,139 | $32 | +42.3% |
| Retail / General Commercial | 1,049 | $168 | +14.0% |
| Condominium (landlots) | 430 | $906 | **+8.8%** |
| Industrial | 1,029 | $75 | +8.1% |
| Mixed Use | 2,050 | $71 | +7.7% |
| Small Multi-Family (2–4 units) | 10,611 | $23 | +3.6% |
| Single Family Residential | 69,982 | $10 | **+2.3%** |
| Agricultural | 3 | −$12 | −0.5% |
| Townhome / Rowhouse | 10,497 | −$3 | **−0.8%** |
| Large Multi-Family (5+ units) | 1,623 | −$26 | −1.0% |
| Office / Commercial Condo | 623 | −$54 | −3.4% |
| Hotel | 104 | −$1,705 | **−16.4%** |

### Census-quintile findings

After de-pinning the condo-imputation artifact, the income and minority quintile charts show the actual structural shift:

- **Income quintile** (excl. vacant): Q1 (lowest income) **−6.1%**, Q2 +1%, Q3 +4.1%, Q4 +3.4%, Q5 (highest) **+10.1%**. The lowest-income quintile sees a tax cut, the highest sees the largest increase — classic LVT progressivity by neighborhood income.
- **Minority quintile** (excl. vacant): Q1 (lowest minority share) +3.7%, Q2 +6.7%, Q3 +5.7%, Q4 +4.1%, Q5 (highest minority share) **−9.4%**. The highest-minority quintile sees a meaningful tax cut.

### Why most residential medians sit slightly above zero

The break-even land share for a revenue-neutral 4:1 split equals the **city's value-weighted taxable land share** — for Pittsburgh, **25.3%** after the collapse + imputation. Parcels with land share above 25.3% pay more under the split; below pay less. The non-condo improvement ratio is 0.73 (≈27% land share), so typical residential parcels sit just above the break-even.

Single Family and Small Multi-Family medians come out at +2.3% and +3.6%, with substantial within-category spread — about 24% of single-family parcels still see a >10% *decrease*. Townhome, Large Multi-Family, Office, and Hotel medians come out negative. This is the genuine city outcome under Pittsburgh's stale 2012 base-year assessment, not a modeling artifact.

### Caveats not captured

- **Act 77 senior tax relief** (30–40% city-tax discount for income-restricted senior owner-occupants) is not modeled; the parcel file lacks income/age flags. Affects perhaps 5–10% of homestead parcels.
- **LERTA / Act 42 city building abatements** are not separately flagged (the `ABATEMENTFLAG` column only marks the 175 parcels with *county*-level abatements). City abatements affect a small share of new-construction and revitalization parcels.
- **Condo land imputation** uses the neighborhood's (then ward's) non-condo median IR as a proxy. Stacked condos genuinely have lower per-unit land share than detached houses, so this approach probably slightly over-allocates land to condo landlots and overstates their LVT burden. The treatment is now at the landlot level — one tax bill per building — matching how the building is one physical site.
